In [ ]:
# ===== Hakeem Benchmark - T323 =====

!pip install -q transformers datasets accelerate bitsandbytes huggingface_hub

from huggingface_hub import login
from google.colab import userdata

try:
    login(token=userdata.get('HF_TOKEN'))
    print("✅ Successfully logged into Hugging Face!")
except Exception:
    print("⚠️ WARNING: HF_TOKEN not found. Gated models will FAIL.")

from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
import torch, time, json, gc, re

# ===== 1. Config =====
QN = 100

MODELS = {
#    "Qwen3-4B":    "Qwen/Qwen3-4B",
#    "Gemma3-4B":   "unsloth/gemma-3-4b-it",
#    "MedGemma":    "unsloth/medgemma-4b-it",
#    "MedGemma-FT": "2kfi/MedGemma-4B-it-finetuned_V2.0",
    "MedGemma-FT-2": "2kfi/Hakeem-MedGemma-4B-MedQA-v1",
}

# It's highly recommended to use the chat template for all instruct/fine-tuned models
CHAT_TEMPLATE_MODELS = {#"Qwen3-4B",
#                        "Gemma3-4B",
#                        "MedGemma",
#                        "MedGemma-FT",
                        "MedGemma-FT-2"}

# ===== 2. Dataset =====
ds = load_dataset("GBaker/MedQA-USMLE-4-options", split=f"test[:{QN}]")

def format_question(example):
    q = example["question"]
    options = "\n".join([f"{k}) {v}" for k, v in example["options"].items()])
    answer = example["answer_idx"]
    return q, options, answer

# ===== 3. Helpers =====
def run_inference(model, tokenizer, prompt, max_new_tokens, use_chat_template=False):
    if use_chat_template:
        messages = [{"role": "user", "content": prompt}]
        # Render template as string first (copied from your working script)
        input_text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
        # Tokenize the string to properly generate both input_ids and attention_mask
        inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
    else:
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    input_length = inputs.input_ids.shape[1]

    start = time.time()
    with torch.no_grad():
        output = model.generate(
            **inputs, # Unpacks both input_ids and attention_mask cleanly
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            temperature=None,
            top_p=None,
        )
    elapsed = time.time() - start

    # Slice out only the newly generated tokens
    generated = output[0][input_length:]
    text = tokenizer.decode(generated, skip_special_tokens=True).strip()
    return text, round(elapsed, 3)

def extract_letter(text):
    text = text.strip()
    patterns = [
        r'\bAnswer[:\s]+([A-D])\b',
        r'\b([A-D])[\.)\s]',
        r'^\(?([A-D])\)?',
        r'([A-D])$',
    ]
    for pat in patterns:
        m = re.search(pat, text, re.IGNORECASE)
        if m:
            return m.group(1).upper()
    m = re.search(r'[A-D]', text, re.IGNORECASE)
    return m.group(0).upper() if m else "?"

# ===== 4. Test Function =====
def test_model(model_name, model_id):
    print(f"\n🔬 Testing: {model_name}")
    use_tmpl = model_name in CHAT_TEMPLATE_MODELS

    tokenizer = AutoTokenizer.from_pretrained(model_id)

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        device_map="auto",
        dtype=torch.bfloat16 # <-- Switched to bfloat16 to match your working script
    )
    model.eval() # <-- Added evaluation mode

    correct, total = 0, 0
    question_logs = []

    for i, example in enumerate(ds):
        q, options, answer = format_question(example)

        mcq_prompt = f"""You are a medical assistant. Answer with only the letter (A, B, C, or D).

Question: {q}
{options}

Answer:"""

        raw_mcq, mcq_time = run_inference(model, tokenizer, mcq_prompt, max_new_tokens=10, use_chat_template=use_tmpl)
        predicted = extract_letter(raw_mcq)
        is_correct = (predicted == answer.upper())
        if is_correct:
            correct += 1
        total += 1

        gen_prompt = f"""You are a medical assistant. For the following question, first state the correct answer letter, then explain your clinical reasoning in 2-3 sentences.

Question: {q}
{options}

Answer:"""
        raw_gen, gen_time = run_inference(model, tokenizer, gen_prompt, max_new_tokens=150, use_chat_template=use_tmpl)

        question_logs.append({
            "index": i,
            "question": q,
            "options": dict(example["options"]),
            "gold_answer": answer.upper(),
            "mcq": {
                "raw_output": raw_mcq,
                "predicted_letter": predicted,
                "correct": is_correct,
                "time_s": mcq_time,
            },
            "generative": {
                "raw_output": raw_gen,
                "time_s": gen_time,
            },
        })

        if (i + 1) % 10 == 0:
            print(f"  [{i+1}/{QN}] Running accuracy: {correct}/{i+1} ({round(correct/(i+1)*100,1)}%)")

    # Cleanup to prevent OOM when switching models
    del model, tokenizer
    torch.cuda.empty_cache()
    gc.collect()

    mcq_times = [q["mcq"]["time_s"] for q in question_logs]
    gen_times  = [q["generative"]["time_s"] for q in question_logs]

    return {
        "model": model_name,
        "model_id": model_id,
        "accuracy": round(correct / total * 100, 1) if total > 0 else 0,
        "correct": correct,
        "total": total,
        "avg_mcq_time_s": round(sum(mcq_times) / len(mcq_times), 3) if mcq_times else 0,
        "avg_gen_time_s": round(sum(gen_times)  / len(gen_times),  3) if gen_times else 0,
        "questions": question_logs,
    }

# ===== 5. Run =====
results = []
for name, model_id in MODELS.items():
    try:
        r = test_model(name, model_id)
        results.append(r)
        print(f"✅ {name}: {r['accuracy']}% ({r['correct']}/{r['total']})")
    except Exception as e:
        print(f"❌ {name}: Failed — {e}")

# ===== 6. Summary Table =====
print("\n" + "="*65)
print(f"{'Model':<20} {'Accuracy':>8} {'Correct/Total':>15} {'MCQ(s)':>8} {'Gen(s)':>8}")
print("="*65)
for r in results:
    print(
        f"{r['model']:<20} {r['accuracy']:>7}% "
        f"{str(r['correct'])+'/'+str(r['total']):>15} "
        f"{r['avg_mcq_time_s']:>7}s "
        f"{r['avg_gen_time_s']:>7}s"
    )

# ===== 7. Save =====
with open("hakeem_results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
print("\n💾 Results saved to hakeem_results.json")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.8 MB/s eta 0:00:00


In [ ]:
from google.colab import files

# Download the file
files.download('hakeem_results.json')
